# 🗺️ Enriquecimiento de Redpiso: Grupo de Control con GIS Medioambiental

## 🎯 Objetivo del Cuaderno
Este *notebook* prepara el *dataset* alternativo extraído del portal Redpiso. El objetivo de esta base de datos es servir como "Grupo de Control" en la fase de modelado para demostrar empíricamente el valor de la Inteligencia Artificial (presente solo en Tecnocasa).

Para que la comparativa sea exhaustiva, se enriquece este conjunto de datos con **Ingeniería de Datos Espaciales (GIS) avanzada**. Se calculan variables geográficas oficiales de impacto acústico y medioambiental (distancia a grandes aglomeraciones, carreteras principales, líneas de ferrocarril y aeropuertos). Si, a pesar de este dopaje espacial, el modelo rinde peor que el de Tecnocasa, quedará demostrada la superioridad predictiva de la extracción de características visuales mediante IA.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import unidecode

In [2]:
# Dataset
ruta_redpiso = "../DATOS/indice_redpiso.xlsx"
df = pd.read_excel(ruta_redpiso)

In [ ]:
print("⚙️ INICIANDO PREPARACIÓN DE REDPISO...")

# 0. CARGAR DATOS
df_redpiso = pd.read_excel("../DATOS/indice_redpiso.xlsx")

# ==============================================================================
# 1. EXTRACCIÓN DE LA LOCALIZACIÓN DESDE EL TÍTULO (LIMPIO Y SIMPLE)
# ==============================================================================
def extraer_zona(titulo):
    if pd.isna(titulo): return ""
    titulo_str = str(titulo).upper().strip()
    
    # Limpiamos prefijos molestos
    prefijos = [
        "PISO EN VENTA EN ", "CHALET EN VENTA EN ", "CASA EN VENTA EN ", 
        "DÚPLEX EN VENTA EN ", "ÁTICO EN VENTA EN ", "INMUEBLE EN VENTA EN ", "ESTUDIO EN VENTA EN ",
        "VENTA EN "
    ]
    for p in prefijos:
        if titulo_str.startswith(p):
            titulo_str = titulo_str.replace(p, "")
            break
            
    # Limpiamos coletillas finales
    ubicacion = titulo_str.replace(", MADRID, MADRID", "").replace(", MADRID", "")
    return ubicacion.strip()

df_redpiso['Direccion_Extraida'] = df_redpiso['titulo'].apply(extraer_zona)

# Diccionario COMPLETO de Distritos Oficiales
distritos_madrid = [
    "CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
    "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
    "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA",
    "VILLAVERDE", "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"
]

def asignar_clave_renta(direccion):
    dir_limpia = unidecode.unidecode(direccion)
    
    # 1. Buscamos si es un distrito
    for distrito in distritos_madrid:
        if distrito in dir_limpia:
            return distrito
            
    # 2. Si no es distrito, extraemos el municipio (lo que hay tras la coma)
    partes = dir_limpia.split(",")
    posible_lugar = partes[-1].strip()
    
    # Limpiamos basura (si es muy corto o es una calle)
    if len(posible_lugar) <= 3 or "CALLE" in posible_lugar or posible_lugar in ["LAS", "LOS", "EL", "LA"]:
        return None # Devolvemos None para luego borrar esta fila
        
    return posible_lugar

df_redpiso['Localizacion_Clave'] = df_redpiso['Direccion_Extraida'].apply(asignar_clave_renta)

# ELIMINAMOS LA BASURA (Inmuebles sin localización válida)
df_redpiso = df_redpiso.dropna(subset=['Localizacion_Clave'])


# ==============================================================================
# 2. CRUCE CON LA RENTA NETA MEDIA
# ==============================================================================
print("💶 Cruzando datos de Renta Neta Media...")

# Obligamos a Pandas a leer el Total como texto para no perder los ceros
df_rentas = pd.read_csv("../DATOS/rentaMedia.csv", sep=";", dtype={'Total': str})

df_rentas["Municipio_Limpio"] = df_rentas["Municipios"].str.extract(r"\d+\s+(.*)").squeeze().str.upper().str.strip()

if 'Distritos' in df_rentas.columns:
    df_rentas["Distrito_Codigo"] = df_rentas["Distritos"].str.extract(r"(28079\d{2})")
    codigos_distritos = {
        "2807901": "CENTRO", "2807902": "ARGANZUELA", "2807903": "RETIRO",
        "2807904": "SALAMANCA", "2807905": "CHAMARTIN", "2807906": "TETUAN",
        "2807907": "CHAMBERI", "2807908": "FUENCARRAL-EL PARDO", "2807909": "MONCLOA-ARAVACA",
        "2807910": "LATINA", "2807911": "CARABANCHEL", "2807912": "USERA",
        "2807913": "PUENTE DE VALLECAS", "2807914": "MORATALAZ", "2807915": "CIUDAD LINEAL",
        "2807916": "HORTALEZA", "2807917": "VILLAVERDE", "2807918": "VILLA DE VALLECAS",
        "2807919": "VICALVARO", "2807920": "SAN BLAS-CANILLEJAS", "2807921": "BARAJAS"
    }
    df_rentas["Localizacion_Clave"] = df_rentas["Distrito_Codigo"].map(codigos_distritos).fillna(df_rentas["Municipio_Limpio"])
else:
    df_rentas["Localizacion_Clave"] = df_rentas["Municipio_Limpio"]

# Filtrar solo "Renta neta media por hogar" para evitar duplicados
df_rentas = df_rentas[df_rentas['Indicadores de renta media y mediana'] == 'Renta neta media por hogar']
# Cogemos el año máximo disponible (2022)
df_rentas = df_rentas[df_rentas['Periodo'] == df_rentas['Periodo'].max()]

# Al borrar el punto ahora a "38.060" (texto), se quedará en 38060 correcto
df_rentas['Renta_Neta_Media'] = pd.to_numeric(df_rentas['Total'].str.replace('.', ''), errors='coerce')

# Cruce final
df_redpiso = df_redpiso.merge(df_rentas[["Localizacion_Clave", "Renta_Neta_Media"]].drop_duplicates('Localizacion_Clave'), on="Localizacion_Clave", how="left")

# (OPCIONAL) Limpieza final de nulos si quedase alguno
df_redpiso = df_redpiso.dropna(subset=['Renta_Neta_Media'])

⚙️ INICIANDO PREPARACIÓN DE REDPISO...
💶 Cruzando datos de Renta Neta Media...


In [4]:
print("\n🔍 INICIANDO AUDITORÍA DEL CRUCE DE RENTAS EN REDPISO...\n")

# 1. ¿Cuántos han usado la Renta Real vs la Mediana?
mediana_renta = df_redpiso['Renta_Neta_Media'].median()
pisos_con_mediana = (df_redpiso['Renta_Neta_Media'] == mediana_renta).sum()
pisos_reales = len(df_redpiso) - pisos_con_mediana
porcentaje_exito = (pisos_reales / len(df_redpiso)) * 100

print(f"📊 1. COBERTURA DE RENTA:")
print(f"   - Total de inmuebles RedPiso: {len(df_redpiso)}")
print(f"   - Cruzados con éxito (Renta Real): {pisos_reales} ({porcentaje_exito:.1f}%)")
print(f"   - Fallidos (Rellenados con mediana): {pisos_con_mediana}\n")

# 2. Análisis de las Localizaciones que han fallado
if pisos_con_mediana > 0:
    print("⚠️ 2. ZONAS QUE NO HAN CRUZADO BIEN (Top 5 más frecuentes):")
    # Buscamos los que tienen la renta mediana exacta
    fallos = df_redpiso[df_redpiso['Renta_Neta_Media'] == mediana_renta]
    print(fallos['Localizacion_Clave'].value_counts())
    print("\n   *Nota: Revisa si los nombres de arriba están escritos distinto en el Excel del INE.")
    print("-" * 50)

# 3. Prueba de realidad (Comprobar coherencia económica)
print("\n🏆 3. PRUEBA DE REALIDAD (Rentas asignadas):")
try:
    print(f"   - Renta Mínima detectada: {df_redpiso['Renta_Neta_Media'].min():,.0f} €")
    print(f"   - Renta Media del dataset: {df_redpiso['Renta_Neta_Media'].mean():,.0f} €")
    print(f"   - Renta Máxima detectada: {df_redpiso['Renta_Neta_Media'].max():,.0f} €\n")
    
    print("   Zonas con MAYOR renta asignada:")
    display(df_redpiso[['Localizacion_Clave', 'Renta_Neta_Media']].drop_duplicates().sort_values(by='Renta_Neta_Media', ascending=False).head(3))
    
    print("\n   Zonas con MENOR renta asignada:")
    display(df_redpiso[['Localizacion_Clave', 'Renta_Neta_Media']].drop_duplicates().sort_values(by='Renta_Neta_Media', ascending=True).head(3))
except Exception as e:
    print("Error mostrando estadísticas:", e)


🔍 INICIANDO AUDITORÍA DEL CRUCE DE RENTAS EN REDPISO...

📊 1. COBERTURA DE RENTA:
   - Total de inmuebles RedPiso: 2829
   - Cruzados con éxito (Renta Real): 2524 (89.2%)
   - Fallidos (Rellenados con mediana): 305

⚠️ 2. ZONAS QUE NO HAN CRUZADO BIEN (Top 5 más frecuentes):
Localizacion_Clave
CENTRO    305
Name: count, dtype: int64

   *Nota: Revisa si los nombres de arriba están escritos distinto en el Excel del INE.
--------------------------------------------------

🏆 3. PRUEBA DE REALIDAD (Rentas asignadas):
   - Renta Mínima detectada: 29,050 €
   - Renta Media del dataset: 44,556 €
   - Renta Máxima detectada: 92,596 €

   Zonas con MAYOR renta asignada:


,Localizacion_Clave,Renta_Neta_Media
554,POZUELO DE ALARCON,92596
617,BOADILLA DEL MONTE,82812
164,CHAMARTIN,74842



   Zonas con MENOR renta asignada:


,Localizacion_Clave,Renta_Neta_Media
186,CADALSO DE LOS VIDRIOS,29050
1558,TIELMES,30000
190,PUENTE DE VALLECAS,30339


In [5]:
# ==============================================================================
# AUDITORÍA Y LISTADO FINAL (Imprime todas las zonas y su renta)
# ==============================================================================
print("\n📋 LISTADO DE LOCALIZACIONES Y SUS RENTAS ASIGNADAS:\n")

# Mostramos absolutamente todas las filas
pd.set_option('display.max_rows', None)

resumen_rentas = df_redpiso[['Localizacion_Clave', 'Renta_Neta_Media']].drop_duplicates().sort_values('Localizacion_Clave')
# Formatear bonito para que se lea fácil
resumen_rentas['Renta_Neta_Media'] = resumen_rentas['Renta_Neta_Media'].apply(lambda x: f"{x:,.0f} €" if pd.notna(x) else "¡FALLO NULO!")

display(resumen_rentas)

pd.reset_option('display.max_rows')

# Check de nulos finales
nulos = df_redpiso['Renta_Neta_Media'].isna().sum()
print(f"\n✅ Total Inmuebles Válidos: {len(df_redpiso)}")
print(f"⚠️ Inmuebles que han quedado sin Renta (Nulos): {nulos}")


📋 LISTADO DE LOCALIZACIONES Y SUS RENTAS ASIGNADAS:



,Localizacion_Clave,Renta_Neta_Media
7,ALCALA DE HENARES,"39,439 €"
866,ALCOBENDAS,"62,298 €"
2,ALCORCON,"39,129 €"
315,ARANJUEZ,"38,060 €"
258,ARGANDA DEL REY,"37,574 €"
146,ARGANZUELA,"49,236 €"
22,ARROYOMOLINOS,"51,310 €"
422,BARAJAS,"54,343 €"
617,BOADILLA DEL MONTE,"82,812 €"
186,CADALSO DE LOS VIDRIOS,"29,050 €"



✅ Total Inmuebles Válidos: 2829
⚠️ Inmuebles que han quedado sin Renta (Nulos): 0


In [6]:
df_redpiso

,url,titulo,referencia,precio,precio_num,superficie_m2,habitaciones,banos,estado,calificacion_energetica,...,piscina,amueblado,cocina_equipada,agua_caliente,orientacion,tipo_fachada,garajes,Direccion_Extraida,Localizacion_Clave,Renta_Neta_Media
0,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en CALLE VEREDA DEL PRADO, Ciemp...",RP1972024125050,115.000 €,115000.0,88.0,3.0,1.0,RESERVADO,E,...,False,False,False,NaN,NaN,NaN,NaN,"CALLE VEREDA DEL PRADO, CIEMPOZUELOS",CIEMPOZUELOS,34497
1,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en Villaverde, Madrid, Madrid",RP472026145978,148.000 €,148000.0,38.0,1.0,1.0,DISPONIBLE,F,...,False,False,False,NaN,NaN,NaN,NaN,VILLAVERDE,VILLAVERDE,32527
2,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en Alcorcón, Madrid",RP1692024122535,159.000 €,159000.0,68.0,3.0,1.0,RESERVADO,F,...,False,False,False,Gas natural,NaN,NaN,NaN,ALCORCÓN,ALCORCON,39129
3,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en La Fortuna, Leganés, Madrid",RP1312025142220,164.900 €,164900.0,50.0,2.0,1.0,DISPONIBLE,G,...,False,False,False,Individual,Sureste,NaN,NaN,"LA FORTUNA, LEGANÉS",LEGANES,36260
4,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en San Blas-Canillejas, Madrid, ...",RP492026145360,172.260 €,172260.0,57.0,3.0,1.0,DISPONIBLE,G,...,False,False,True,NaN,NaN,Ladrillo,NaN,SAN BLAS-CANILLEJAS,SAN BLAS-CANILLEJAS,41474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2824,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en CALLE MELQUIADES BIENCINTO, S...",RP152025137910,A consultar,NaN,40.0,2.0,1.0,DISPONIBLE,G,...,False,False,True,NaN,NaN,NaN,NaN,"CALLE MELQUIADES BIENCINTO, SAN DIEGO, PUENTE ...",PUENTE DE VALLECAS,30339
2825,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en CALLE FELISA MENDEZ, San Dieg...",RP152025140165,A consultar,NaN,75.0,2.0,1.0,DISPONIBLE,G,...,False,False,False,NaN,NaN,Ladrillo,NaN,"CALLE FELISA MENDEZ, SAN DIEGO, PUENTE DE VALL...",PUENTE DE VALLECAS,30339
2826,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en CALLE HACHERO, San Diego, Pue...",RP152025140863,A consultar,NaN,40.0,1.0,1.0,DISPONIBLE,G,...,False,False,False,NaN,NaN,NaN,NaN,"CALLE HACHERO, SAN DIEGO, PUENTE DE VALLECAS",PUENTE DE VALLECAS,30339
2827,https://www.redpiso.es/inmueble/piso-en-venta-...,"Piso en venta en CALLE PUERTO DE MADERI, San D...",RP152025141825,A consultar,NaN,80.0,2.0,2.0,DISPONIBLE,G,...,False,False,False,NaN,NaN,Ladrillo,NaN,"CALLE PUERTO DE MADERI, SAN DIEGO, PUENTE DE V...",PUENTE DE VALLECAS,30339


In [ ]:
# ==============================================================================
# 3. GEOCODIFICACIÓN (OBTENER LATITUD Y LONGITUD)
# ==============================================================================
print("🌍 Iniciando Geocodificación para obtener coordenadas GPS...")
# Usamos tqdm para ver la barra de progreso porque esto tarda un poco
tqdm.pandas()

geolocator = Nominatim(user_agent="tfg_inmobiliario_bot")
# RateLimiter para no saturar la API gratuita (1 segundo entre peticiones)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def obtener_coordenadas(direccion):
    try:
        # Añadimos "Comunidad de Madrid, Spain" para acotar la búsqueda
        busqueda = f"{direccion}, Comunidad de Madrid, Spain"
        location = geolocator.geocode(busqueda, timeout=10)
        if location:
            return pd.Series((location.latitude, location.longitude))
        else:
            # Si falla con la calle, intentamos solo con el municipio/barrio
            municipio = direccion.split(",")[-1]
            loc_municipio = geolocator.geocode(f"{municipio}, Comunidad de Madrid, Spain", timeout=10)
            if loc_municipio:
                return pd.Series((loc_municipio.latitude, loc_municipio.longitude))
            return pd.Series((np.nan, np.nan))
    except:
        return pd.Series((np.nan, np.nan))

# Aplicamos la función (Tomará un rato si tienes muchos pisos)
df_redpiso[['Latitud', 'Longitud']] = df_redpiso['Direccion_Extraida'].progress_apply(obtener_coordenadas)

print(f"✅ Coordenadas recuperadas. Nulos restantes: {df_redpiso['Latitud'].isna().sum()}")

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

print("🔊 Iniciando cálculo de Ruido y Distancias (GeoPandas)...")

# 1. ELIMINAR NULOS (Vital para GeoPandas)
# Si algún piso no se pudo geocodificar (nulo en Latitud), lo descartamos temporalmente o lo filtramos
# para que no rompa el cálculo espacial.
df_redpiso_geo = df_redpiso.dropna(subset=['Latitud', 'Longitud']).copy()

# 2. CREAR EL GEODATAFRAME
df_redpiso_geo = gpd.GeoDataFrame(
    df_redpiso_geo, 
    geometry=gpd.points_from_xy(df_redpiso_geo.Longitud, df_redpiso_geo.Latitud), 
    crs="EPSG:4326"
)

# 3. CARGAR SHAPEFILES (Asegúrate de que las rutas son las correctas en tu PC)
print("   -> Cargando mapas de ruido del Ministerio...")
ruido_carreteras = gpd.read_file("../DATOS/mer-carreteras_lden/CDERUI02_Lden_RepGeo_LocalId_C.shp")
ruido_ferroviario = gpd.read_file("../DATOS/mer-lineas_ferroviarias_lden/CDERUI02_Lden_RepGeo_LocalId_F.shp")
ruido_aglomeraciones = gpd.read_file("../DATOS/mer-aglomeraciones_lden/CDERUI02_Lden_RepGeo_LocalId_Ag.shp")
ruido_aeropuertos = gpd.read_file("../DATOS/mer-aeropuertos_lden/CDERUI02_Lden_RepGeo_LocalId_A.shp")

# 4. REPROYECTAR A METROS (UTM zona 30N)
proj_crs = "EPSG:25830"
df_redpiso_geo = df_redpiso_geo.to_crs(proj_crs)
ruido_carreteras = ruido_carreteras.to_crs(proj_crs)
ruido_ferroviario = ruido_ferroviario.to_crs(proj_crs)
ruido_aglomeraciones = ruido_aglomeraciones.to_crs(proj_crs)
ruido_aeropuertos = ruido_aeropuertos.to_crs(proj_crs)

# 5. APLICAR UMBRALES DE DISTANCIA
umbral_carretera = 1000     # metros
umbral_ferrocarril = 1500   # metros
umbral_aglomeracion = 1500  # metros
umbral_aeropuerto = 3000    # metros

print("   -> Calculando distancias más cercanas (sjoin_nearest)...")
# ---- JOIN NEAREST 1: Carreteras
temp = df_redpiso_geo.sjoin_nearest(ruido_carreteras[['geometry', 'Lden']], how='left', distance_col='dist_a_carretera', max_distance=umbral_carretera)
df_redpiso_geo = temp[~temp.index.duplicated(keep='first')].copy()
df_redpiso_geo.rename(columns={'Lden': 'ruido_carretera'}, inplace=True)
df_redpiso_geo.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 2: Ferroviario
temp = df_redpiso_geo.sjoin_nearest(ruido_ferroviario[['geometry', 'Lden']], how='left', distance_col='dist_a_ferrocarril', max_distance=umbral_ferrocarril)
df_redpiso_geo = temp[~temp.index.duplicated(keep='first')].copy()
df_redpiso_geo.rename(columns={'Lden': 'ruido_ferrocarril'}, inplace=True)
df_redpiso_geo.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 3: Aglomeraciones
temp = df_redpiso_geo.sjoin_nearest(ruido_aglomeraciones[['geometry', 'Lden']], how='left', distance_col='dist_a_aglomeracion', max_distance=umbral_aglomeracion)
df_redpiso_geo = temp[~temp.index.duplicated(keep='first')].copy()
df_redpiso_geo.rename(columns={'Lden': 'ruido_urbano'}, inplace=True)
df_redpiso_geo.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 4: Aeropuertos
temp = df_redpiso_geo.sjoin_nearest(ruido_aeropuertos[['geometry', 'Lden']], how='left', distance_col='dist_a_aeropuerto', max_distance=umbral_aeropuerto)
df_redpiso_geo = temp[~temp.index.duplicated(keep='first')].copy()
df_redpiso_geo.rename(columns={'Lden': 'ruido_aeropuerto'}, inplace=True)
df_redpiso_geo.drop(columns=['index_right'], inplace=True, errors='ignore')

# 6. LIMPIEZA DE DISTANCIAS Y RUIDOS
print("   -> Ajustando umbrales y calculando promedios...")
df_redpiso_geo.loc[df_redpiso_geo['dist_a_carretera'] > umbral_carretera, 'ruido_carretera'] = np.nan
df_redpiso_geo.loc[df_redpiso_geo['dist_a_ferrocarril'] > umbral_ferrocarril, 'ruido_ferrocarril'] = np.nan
df_redpiso_geo.loc[df_redpiso_geo['dist_a_aglomeracion'] > umbral_aglomeracion, 'ruido_urbano'] = np.nan
df_redpiso_geo.loc[df_redpiso_geo['dist_a_aeropuerto'] > umbral_aeropuerto, 'ruido_aeropuerto'] = np.nan

ruido_minimo = 30

for col in ['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']:
    df_redpiso_geo[col] = df_redpiso_geo[col].replace(0.0, np.nan)
    df_redpiso_geo[col] = df_redpiso_geo[col].fillna(ruido_minimo)

df_redpiso_geo['ruido_promedio'] = df_redpiso_geo[['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']].mean(axis=1)
df_redpiso_geo['ruido_promedio'] = df_redpiso_geo['ruido_promedio'].fillna(ruido_minimo)

df_redpiso_geo['dist_a_carretera'] = df_redpiso_geo['dist_a_carretera'].fillna(umbral_carretera + 100)
df_redpiso_geo['dist_a_ferrocarril'] = df_redpiso_geo['dist_a_ferrocarril'].fillna(umbral_ferrocarril + 100)
df_redpiso_geo['dist_a_aglomeracion'] = df_redpiso_geo['dist_a_aglomeracion'].fillna(umbral_aglomeracion + 100)
df_redpiso_geo['dist_a_aeropuerto'] = df_redpiso_geo['dist_a_aeropuerto'].fillna(umbral_aeropuerto + 100)

# 7. RETORNO AL FORMATO NORMAL Y GUARDADO
df_redpiso_geo = df_redpiso_geo.to_crs("EPSG:4326")
df_redpiso_final = pd.DataFrame(df_redpiso_geo.drop(columns=['geometry']))

print("✅ ¡Misión cumplida! Tu dataset de RedPiso ya tiene Renta, Distancias y Ruido.")

# Si quieres guardarlo para no tener que repetir la geolocalización y los mapas:
#df_redpiso_final.to_csv("../DATOS/redpiso_enriquecido.csv", index=False)

In [7]:
#df_redpiso.to_parquet("../DATOS/redpiso_RentaNetaMedia.parquet", index=False)